In [20]:
"""

Bibliotecas utilizadas para a raspagem de dados (mais informações estão disponíveis no READ.md)

"""

import requests
from bs4 import BeautifulSoup
from datetime import datetime, date, time
import dateparser
import re
import json

In [21]:
"""

Sessão para testar a abertura de url's diferentes 

"""

url = "https://informacoes.anatel.gov.br/legislacao/resolucoes/2000/563-resolucao-252"
url2 = "https://informacoes.anatel.gov.br/legislacao/leis/473-lei-9295"
url3 = "https://informacoes.anatel.gov.br/legislacao/resolucoes/2012/191-resolucao-589"
resposta = requests.get(url)

print(resposta.status_code)

print(resposta.text[:100])

200

<!DOCTYPE html>
<!--[if lt IE 7]>      <html class="no-js lt-ie9 lt-ie8 lt-ie7" lang="pt-br" dir="l


In [22]:
class Documento:
    """
    
    Classe principal Documento, ela abrange todas as funções que serão necessárias para a raspagem de dados.Além disso, a classe
    possui um construtor capaz de extrair informações de uma url bem como extrair as mesmas informações a partir de um JSON passado.
    
    """
    def __init__(self,resolucao):
        """
    
        Como em python não é possível fazer sobrecarga de métodos propriamente, o construtor verifica se a resolução passada é uma url por meio do uso
        de expressões regulares, caso contrário a classe entende que a resolução está no formato JSON.
    
        """
        if self.is_url(resolucao):
            resposta = requests.get(resolucao)
            soup = BeautifulSoup(resposta.text, "html.parser")
            self.title=soup.title.string
            self.data= self.get_published(soup)
            self.revoked= self.revogation()
            self.text=self.get_text(soup)
        else:      
            with open(resolucao, "r", encoding="utf-8") as f:
                json_doc=json.load(f)      
            self.title= json_doc[0]["Título"]
            self.data= datetime.fromisoformat(json_doc[0]["Data de Publicação"])
            self.revoked= json_doc[0]["Revogada"]
            self.text= json_doc[0]["Texto"]


    def is_url(self, resolucao):
        """
    
        Utiliza expressões regulares para retornar um booleano que indica que se a resolução é uma url.
    
        """
        if re.match(r"https?://", resolucao):
            return True
        return False

    def get_published(self,soup):
        """

        Retorna a data de publicação da resolução a partir da classe em HTML "documentPublished" no formato datetime.
    
        """
        data=soup.find(class_="documentPublished")
        data=data.get_text(strip=True)
        data=data[11:]
        return dateparser.parse(data)

    def revogation(self):
        """

        Retorna um booleano para indicar se a resolução foi revogada buscando no próprio título por essa informação
    
        """
        if "(REVOGADA)" in self.title:
            return True
        else:
            return False

    def redacao(self,i,p,paragrafos):
        """

        Retorna a informação se a linha atual sofreu uma nova redação(Ainda está sendo propriamente desenvolvida)

        """
        texto = p.get_text(strip=True)
        prox=i+1
        if prox>=len(paragrafos):
            return False
        texto_proximo= paragrafos[prox].get_text(strip=True)
        redacao_regex = re.compile(r"\(Redação.*")
        style = p.get("style", "")   
        if "line-through" in style:
            redacao_match = redacao_regex.search(texto_proximo) 
            return True
        else:
            return False

    def get_itens(self,documento,i,linhas,artigo_atual):
        """

        Cria um loop que irá analisar todos os itens dentro de um artigo e juntá-los ao artigo correspondente 
        (deve abranger ainda letras mas está em desenvolvimento). Além disso, a função conta quantas linhas 
        foram usadas para que elas possam ser puladas.

        """
        sentinel_itens=False
        contador_itens=0
        while sentinel_itens == False:
            if i+contador_itens +1 >= len(linhas):
                texto_atual=linhas[i+contador_itens].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["itens"].append(texto_atual)
                break
            else:
                texto_atual=linhas[i+contador_itens].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["itens"].append(texto_atual)
                texto_proximo = linhas[i+1+contador_itens].get_text(strip=True)
                texto_proximo = texto_proximo.replace(u'\u00A0', ' ')
                if re.match(r"Art\.\s*\d+(?:-[A-Za-z])?", texto_proximo) or re.match(r"§\s*\d+", texto_proximo):
                    sentinel_itens=True
                else:
                    contador_itens+=1
        skip=contador_itens
        return skip

    def get_paragrafos(self,documento,i,linhas,artigo_atual):
        """

        Cria um loop que irá analisar todos os parágrafos dentro de um artigo, bem como os itens desses parágrafos e juntá-los
        ao artigo correspondente (deve abranger ainda letras mas está em desenvolvimento).Além disso, a função conta quantas 
        linhas foram usadas para que elas possam ser puladas.

        """
        sentinel_paragrafos=False
        contador_paragrafos=0
        while sentinel_paragrafos == False:
            if i+contador_paragrafos >= len(linhas):
                texto_atual=linhas[i+contador_paragrafos].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["paragrafos"].append(texto_atual)
                break
            else:
                texto_atual=linhas[i+contador_paragrafos].get_text(strip=True)
                texto_atual = texto_atual.replace(u'\u00A0', ' ')
                documento[artigo_atual]["paragrafos"].append(texto_atual)
                texto_proximo = linhas[i+1+contador_paragrafos].get_text(strip=True)
                texto_proximo = texto_proximo.replace(u'\u00A0', ' ')
                if re.match(r"Art\.\s*\d+(?:-[A-Za-z])?", texto_proximo):
                    sentinel_paragrafos=True
                else:
                    contador_paragrafos+=1
        skip=contador_paragrafos
        return skip

    def get_text(self,soup):
        """

        Loop principal que coleta todo o texto da url e o armazena em um dicionário cuja chave é o número do artigo.Além disso,
        caso tenham anexos na resolução a função adiciona "AnexoX" ao nome da chave do artigo correspondente.

        """
        linhas = soup.find_all("p", class_="texto-recuo-1a-linha")
        documento = {}
        artigo_atual = None
        contador=0
        skip=0

        for i,p in enumerate(linhas):
            if skip == 0:
                texto = p.get_text(strip=True)
                texto = texto.replace(u'\u00A0', ' ')
                new_artigo= re.match(r"Art\.\s*\d+(?:-[A-Za-z])?", texto)
                if self.redacao(i,p,linhas):
                        continue
                else:
                    if new_artigo:
                        artigo_atual = new_artigo.group()
                        artigo_atual= artigo_atual.replace(" ","")
                        if artigo_atual not in documento and contador==0:
                            documento[artigo_atual] = {"texto": texto, "itens": [],"paragrafos":[]}
                        elif artigo_atual in documento and artigo_atual == "Art.1" :
                            contador+=1
                            artigo_atual=artigo_atual+"Anexo"+str(contador)
                            documento[artigo_atual] = {"texto": texto, "itens": [],"paragrafos":[]}
                        else:
                            artigo_atual=artigo_atual+"Anexo"+str(contador)
                            documento[artigo_atual] = {"texto": texto, "itens": [],"paragrafos":[]}

                    paragraph_match= re.match(r"§\s*\d+", texto)
                    item_match = re.match(r"[IVXL]", texto)
                    if item_match:
                        skip=self.get_itens(documento,i,linhas,artigo_atual)
            
                    if paragraph_match:
                        skip=self.get_paragrafos(documento,i,linhas,artigo_atual)
 
            else:
                skip-=1        
        return documento    

    def create_json(self):
        """

        Cria o JSON a partir dos metadados de uma resolução separando o JSON em: "Título","Data de Publicação","Revogada","Texto".
        O nome do JSON criado é o mesmo nome do título da resolução.

        """
        data_str = self.data.isoformat()
        info=[]
        info.append({"Título": self.title,"Data de Publicação": data_str,"Revogada": self.revoked,"Texto":self.text})
        json_doc = json.dumps(info, ensure_ascii=False, indent=4)
        with open(f"{self.title}.json", "w", encoding="utf-8") as f:
            json.dump(info, f, ensure_ascii=False, indent=4)
        

In [23]:
"""

Sessão de testes de criação do JSON a partir de uma url.

"""
teste=Documento("https://informacoes.anatel.gov.br/legislacao/leis/473-lei-9295")
print(teste.title)
print(teste.data)
print(teste.revoked)
print(teste.text['Art.1'])
teste.create_json()

teste_2=Documento("https://informacoes.anatel.gov.br/legislacao/resolucoes/2000/563-resolucao-252")
print(teste_2.title)
print(teste_2.data)
print(teste_2.revoked)
print(teste_2.text)
teste_2.create_json()

Anatel - Lei nº 9.295, de 19 de julho de 1996
1996-07-20 09:27:00
False
{'texto': 'Art. 1º A organização dos serviços de telecomunicações, a exploração de Serviço Móvel Celular, de Serviço Limitado e de Serviço de Transporte de Sinais de Telecomunicações por Satélite, bem como a utilização da rede pública de telecomunicações para a prestação de Serviço de Valor Adicionado, regulam-se por esta Lei, relativamente aos serviços que menciona, respeitado o que disciplina a legislação em vigor, em especial aLei n° 4.117, de 27 de agosto de 1962, pelasLeis n° s 8.987, de 13 de fevereiro de 1995, e9.074, de 7 de julho de 1995, e, no que for compatível, pela legislação de telecomunicações, em vigor.(Revogado pela Lei nº 9.472, 16.7.1997)', 'itens': [], 'paragrafos': []}
Anatel - Resolução nº 252, de 20 de dezembro de 2000 (REVOGADA)
2000-12-29 14:04:00
True
{'Art.1': {'texto': 'Art.1º Aprovar o Regulamento de Sinalização para Usuários, na forma do anexo.', 'itens': [], 'paragrafos': []}, 'Art.2'

In [24]:
"""

Sessão de teste da transformação de um JSON nos elementos da classe novamente.

"""
teste_3=Documento("Anatel - Lei nº 9.295, de 19 de julho de 1996.json")
print(teste_3.title)
print(teste_3.data)
print(teste_3.revoked)
print(teste_3.text)

Anatel - Lei nº 9.295, de 19 de julho de 1996
1996-07-20 09:27:00
False
{'Art.1': {'texto': 'Art. 1º A organização dos serviços de telecomunicações, a exploração de Serviço Móvel Celular, de Serviço Limitado e de Serviço de Transporte de Sinais de Telecomunicações por Satélite, bem como a utilização da rede pública de telecomunicações para a prestação de Serviço de Valor Adicionado, regulam-se por esta Lei, relativamente aos serviços que menciona, respeitado o que disciplina a legislação em vigor, em especial aLei n° 4.117, de 27 de agosto de 1962, pelasLeis n° s 8.987, de 13 de fevereiro de 1995, e9.074, de 7 de julho de 1995, e, no que for compatível, pela legislação de telecomunicações, em vigor.(Revogado pela Lei nº 9.472, 16.7.1997)', 'itens': [], 'paragrafos': []}, 'Art.2': {'texto': 'Art. 2° Sujeitam-se à disciplina desta Lei os serviços de telecomunicações elencados no art.1° , observados as seguintes definições:(Revogado pela Lei nº 9.472, 16.7.1997)', 'itens': [], 'paragrafos